# FDRL-IDS: Federated Deep Reinforcement Learning based IDS

**Paper:** "Federated reinforcement learning based intrusion detection system using dynamic attention mechanism"  
**Journal:** Journal of Information Security and Applications 78 (2023) 103608

---

### Pipeline

1. Load & preprocess NSL-KDD (Section 5.1)
2. Train Denoising Autoencoder (Section 4.1)
3. Distribute data among agents (Section 6)
4. Federated training with Dynamic Attention (Algorithms 1-3)
5. Evaluate & visualize (Section 6, Figures 3-8, Table 3)


In [ ]:
# ============================================================
# Cell 1 - Setup: install deps & upload source code
# ============================================================
import os, sys, shutil

# --- On Kaggle, copy source tree into /kaggle/working if needed ---
WORKING = '/kaggle/working'
SRC_DST  = os.path.join(WORKING, 'src')
MAIN_DST = os.path.join(WORKING, 'main_train.py')

# If running from an uploaded dataset that contains the project
PROJECT_CANDIDATES = [
    '/kaggle/input/fdrl-ids',       # if you uploaded as dataset "fdrl-ids"
    '/kaggle/input/fdrl-ids/NT549', # nested zip layout
    '/kaggle/input/nt549',
    '/kaggle/input/nt549/NT549',
    '.',                             # already in working dir
]

project_root = None
for cand in PROJECT_CANDIDATES:
    if os.path.isfile(os.path.join(cand, 'main_train.py')):
        project_root = cand
        break

if project_root and os.path.abspath(project_root) != os.path.abspath(WORKING):
    # Copy src/ and main_train.py into working dir
    for item in ['src', 'main_train.py', 'requirements.txt']:
        src = os.path.join(project_root, item)
        dst = os.path.join(WORKING, item)
        if os.path.exists(src) and not os.path.exists(dst):
            if os.path.isdir(src):
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
    print(f'Copied project from {project_root}')

os.chdir(WORKING)
sys.path.insert(0, WORKING)

# Install requirements
!pip install -q torch numpy pandas scikit-learn matplotlib
print('Setup complete.')

In [ ]:
# ============================================================
# Cell 2 - Imports & device check
# ============================================================
import numpy as np
import pandas as pd
import torch
import time, json, os, sys

sys.path.insert(0, os.getcwd())

from src.data.nsl_kdd import (
    load_nsl_kdd, preprocess_nsl_kdd,
    distribute_data_random, distribute_data_customized
)
from src.models.denoising_autoencoder import (
    DenoisingAutoencoder, train_dae, transform_with_dae
)
from src.federated_learning.orchestrator import FederatedOrchestrator
from src.utils.metrics import compute_metrics, print_metrics, compute_roc_data
from src.utils.visualization import (
    plot_all_training_curves, plot_roc_curve, plot_accuracy_vs_num_agents
)
from src.utils.config import *
from sklearn.preprocessing import MinMaxScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# Cell 3 - Locate NSL-KDD dataset
# ============================================================
DATA_CANDIDATES = [
    '/kaggle/input/nsl-kdd',
    '/kaggle/input/nsl-kdd/NSL-KDD',
    '/kaggle/input/nslkdd',
    '/kaggle/input/nslkdd/NSL-KDD',
    './Dataset/NSL-KDD',
    './Dataset',
]

# Kaggle strips '+' from filenames on upload: KDDTrain+.txt -> KDDTrain.txt
TRAIN_NAMES = ['KDDTrain+.txt', 'KDDTrain.txt']
TEST_NAMES  = ['KDDTest+.txt',  'KDDTest.txt']

data_dir = None
for cand in DATA_CANDIDATES:
    if (any(os.path.isfile(os.path.join(cand, n)) for n in TRAIN_NAMES) and
        any(os.path.isfile(os.path.join(cand, n)) for n in TEST_NAMES)):
        data_dir = cand
        break

if data_dir is None:
    inp = '/kaggle/input'
    if os.path.isdir(inp):
        for d in os.listdir(inp):
            sub = os.path.join(inp, d)
            print(f'{sub}/ -> {os.listdir(sub)[:10]}')
    raise FileNotFoundError(
        'NSL-KDD not found. Expected KDDTrain+.txt or KDDTrain.txt.'
        ' Add NSL-KDD dataset to this notebook.'
    )

train_path = next(
    os.path.join(data_dir, n) for n in TRAIN_NAMES
    if os.path.isfile(os.path.join(data_dir, n))
)
test_path = next(
    os.path.join(data_dir, n) for n in TEST_NAMES
    if os.path.isfile(os.path.join(data_dir, n))
)
print(f'Train: {train_path}  ({os.path.getsize(train_path)/1e6:.1f} MB)')
print(f'Test:  {test_path}  ({os.path.getsize(test_path)/1e6:.1f} MB)')

In [ ]:
# ============================================================
# Cell 4 - Hyperparameters (from paper Section 6)
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Experiment choice: 'random' | 'customized' | 'scalability'
EXPERIMENT  = 'random'
NUM_AGENTS  = 8        # 8 for random, 2 for customized
NUM_ROUNDS  = 30       # Paper: 30 rounds
EPISODES_PER_ROUND = 3 # Paper: 3 episodes per round

# Attention params (Section 6.1)
ATTN_K = 30   # Random: 30   | Customized: 50000
ATTN_A = 50   # Random: 50   | Customized: 200

USE_DAE = True         # Section 4.1 - Denoising Autoencoder
OUTPUT_DIR = '/kaggle/working/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Hyperparameters set.')

## Step 1 - Load & Preprocess NSL-KDD (Section 5.1, Page 7)


In [ ]:
# ============================================================
# Cell 5 - Load and preprocess
# ============================================================
train_df, test_df = load_nsl_kdd(train_path, test_path)
raw_train_df = train_df.copy()  # Keep for customized split

X_train, y_train, X_test, y_test, scaler, feature_dim = preprocess_nsl_kdd(
    train_df, test_df
)
print(f'\nFeature dimension: {feature_dim}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Step 2 - Denoising Autoencoder (Section 4.1, Page 5)


In [ ]:
# ============================================================
# Cell 6 - Train DAE and transform features
# ============================================================
if USE_DAE:
    print('Training Denoising Autoencoder...')
    dae = DenoisingAutoencoder(
        input_dim=feature_dim,
        hidden_dim=DAE_HIDDEN_DIM,
        noise_factor=DAE_NOISE_FACTOR
    )
    dae, dae_losses = train_dae(
        dae, X_train, device,
        epochs=DAE_EPOCHS,
        batch_size=DAE_BATCH_SIZE,
        lr=DAE_LEARNING_RATE
    )
    X_train_dae = transform_with_dae(dae, X_train, device)
    X_test_dae  = transform_with_dae(dae, X_test, device)

    dae_scaler = MinMaxScaler()
    X_train_final = dae_scaler.fit_transform(X_train_dae)
    X_test_final  = dae_scaler.transform(X_test_dae)
    input_dim = X_train_final.shape[1]
    print(f'Feature dim after DAE: {input_dim}')
else:
    X_train_final, X_test_final = X_train, X_test
    input_dim = feature_dim
    print(f'Using raw features, dim={input_dim}')

## Step 3 - Distribute Data Among Agents (Section 6, Pages 8-9)


In [ ]:
# ============================================================
# Cell 7 - Data distribution
# ============================================================
if EXPERIMENT == 'random':
    num_agents = NUM_AGENTS
    agent_data = distribute_data_random(X_train_final, y_train, num_agents, SEED)
    attn_k, attn_a = ATTN_K, ATTN_A

elif EXPERIMENT == 'customized':
    num_agents = 2
    agent_data = distribute_data_customized(
        raw_train_df, y_train, X_train_final, num_agents=2, seed=SEED
    )
    attn_k, attn_a = 50000, 200

print(f'\nExperiment: {EXPERIMENT}, Agents: {num_agents}')
print(f'Attention params: k={attn_k}, a={attn_a}')

## Step 4 - Federated Training (Algorithms 1-3, Section 4, Pages 4-6)


In [ ]:
# ============================================================
# Cell 8 - Create orchestrator & train
# ============================================================
orchestrator = FederatedOrchestrator(
    num_agents=num_agents,
    input_dim=input_dim,
    hidden_layers=DQN_HIDDEN_LAYERS,
    num_actions=NUM_ACTIONS,
    lr=DQN_LEARNING_RATE,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_end=EPSILON_END,
    epsilon_decay=EPSILON_DECAY,
    memory_capacity=MEMORY_CAPACITY,
    batch_size_replay=BATCH_SIZE_REPLAY,
    per_alpha=PER_ALPHA,
    per_beta_start=PER_BETA_START,
    per_beta_end=PER_BETA_END,
    omega=OMEGA,
    dropout=DQN_DROPOUT,
    attention_k=attn_k,
    attention_a=attn_a,
    episodes_per_round=EPISODES_PER_ROUND,
    device=str(device)
)

orchestrator.assign_data(agent_data, test_split_ratio=TEST_SPLIT_RATIO)

print(f'\nStarting federated training: {NUM_ROUNDS} rounds, '
      f'{EPISODES_PER_ROUND} episodes/round ...')
start_time = time.time()

history = orchestrator.train(
    num_rounds=NUM_ROUNDS,
    global_test_X=X_test_final,
    global_test_y=y_test,
    verbose=True
)

total_time = time.time() - start_time
print(f'\nTraining complete in {total_time:.1f}s')

## Step 5 - Final Evaluation (Table 3, Page 11)


In [ ]:
# ============================================================
# Cell 9 - Evaluation
# ============================================================
final_metrics = orchestrator.evaluate_global(X_test_final, y_test)

print('=' * 55)
print('  FINAL RESULTS  (compare with Table 3, Page 11)')
print('=' * 55)
print_metrics(final_metrics, 'Global Model')

print('\n--- Paper Table 3 (NSL-KDD Random) ---')
print('Accuracy=0.9669  FPR=0.0195  Recall=0.9514')
print('Precision=0.9769  F1=0.964  AUC=0.994')

## Step 6 - Visualization (Figures 3-8, Pages 9-13)


In [ ]:
# ============================================================
# Cell 10 - Training curves: accuracy, loss, attention per agent
# ============================================================
plot_dir = os.path.join(OUTPUT_DIR, f'plots_{EXPERIMENT}')
plot_all_training_curves(
    history, num_agents,
    title_suffix=f'(NSL-KDD, {EXPERIMENT} split)',
    save_dir=plot_dir
)

In [ ]:
# ============================================================
# Cell 11 - ROC Curve (Figure 7, Page 13)
# ============================================================
agent0 = orchestrator.agents[0]
agent0.dqn.eval()
probabilities = []
with torch.no_grad():
    for i in range(len(X_test_final)):
        state = torch.FloatTensor(X_test_final[i]).unsqueeze(0).to(device)
        q_vals = agent0.dqn(state)
        probs = torch.softmax(q_vals[0], dim=0)
        probabilities.append(probs[1].item())

fpr_arr, tpr_arr, _ = compute_roc_data(y_test, np.array(probabilities))
plot_roc_curve(
    fpr_arr, tpr_arr, final_metrics['auc_roc'],
    title=f'ROC Curve - NSL-KDD ({EXPERIMENT} split)',
    save_path=os.path.join(plot_dir, 'roc_curve.png')
)

In [ ]:
# ============================================================
# Cell 12 - Save results JSON
# ============================================================
results = {
    'experiment': EXPERIMENT,
    'num_agents': num_agents,
    'num_rounds': NUM_ROUNDS,
    'attention_k': attn_k,
    'attention_a': attn_a,
    'final_metrics': {k: float(v) for k, v in final_metrics.items()},
    'training_time_sec': total_time,
}
results_path = os.path.join(OUTPUT_DIR, f'results_{EXPERIMENT}.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {results_path}')

# Print global accuracy curve
import matplotlib
import matplotlib.pyplot as plt
global_accs = [m['accuracy'] for m in history['global_metrics']]
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(global_accs)+1), global_accs, 'b-o', markersize=4)
plt.xlabel('Federated Round')
plt.ylabel('Global Accuracy')
plt.title(f'Global Model Accuracy vs Round ({EXPERIMENT})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, 'global_accuracy.png'), dpi=150)
plt.show()
plt.close()

---

## (Optional) Scalability Experiment (Section 6.3, Figure 8)

Uncomment and run the cell below to test accuracy vs number of agents.


In [ ]:
# ============================================================
# Cell 13 - Scalability experiment (Section 6.3, Page 11)
# ============================================================
# Set RUN_SCALABILITY = True to execute
RUN_SCALABILITY = False

if RUN_SCALABILITY:
    agent_counts = [2, 4, 6, 8]
    avg_accuracies = []

    for n_agents in agent_counts:
        print(f'\n--- {n_agents} agents ---')
        np.random.seed(SEED)
        torch.manual_seed(SEED)

        ad = distribute_data_random(X_train_final, y_train, n_agents, SEED)
        orch = FederatedOrchestrator(
            num_agents=n_agents, input_dim=input_dim,
            hidden_layers=DQN_HIDDEN_LAYERS, num_actions=NUM_ACTIONS,
            lr=DQN_LEARNING_RATE, gamma=GAMMA,
            epsilon_start=EPSILON_START, epsilon_end=EPSILON_END,
            epsilon_decay=EPSILON_DECAY, memory_capacity=MEMORY_CAPACITY,
            batch_size_replay=BATCH_SIZE_REPLAY,
            per_alpha=PER_ALPHA, per_beta_start=PER_BETA_START,
            per_beta_end=PER_BETA_END, omega=OMEGA, dropout=DQN_DROPOUT,
            attention_k=30, attention_a=50,
            episodes_per_round=EPISODES_PER_ROUND, device=str(device)
        )
        orch.assign_data(ad, test_split_ratio=TEST_SPLIT_RATIO)
        orch.train(num_rounds=10, global_test_X=X_test_final,
                   global_test_y=y_test, verbose=False)
        fm = orch.evaluate_global(X_test_final, y_test)
        avg_accuracies.append(fm['accuracy'])
        print(f'  Accuracy: {fm["accuracy"]:.4f}')

    plot_accuracy_vs_num_agents(
        agent_counts, avg_accuracies,
        title='(NSL-KDD)',
        save_path=os.path.join(OUTPUT_DIR, 'scalability.png')
    )
else:
    print('Scalability experiment skipped. Set RUN_SCALABILITY = True to run.')

---

## (Optional) Customized Split Experiment (Section 6.1, Page 9)

Change `EXPERIMENT='customized'` in Cell 4 and re-run from Cell 5.
